<a href="https://colab.research.google.com/github/danbri/001/blob/master/Factoidal_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Factoidal: Aggregating ClaimReview Factcheck Data from the Web

 - various utilities to pull together CLaimReview sources
 -[ ] using search alerts to find CLaimReview markup, and from there feeds/sitemaps and sites.
 -[ ] IFCN list of signatories - copied to CSV.
  - todo: crawl homepages for sitemaps, feeds.
  - store logos
  - link to Wikidata
 -[ ] Wikidata:
  - run a SPARQL query to pull homepages etc for signatories
 -[ ] Open data from Google: original feed on Data COmmons; also factcheck tools site, API?
 -[ ] Common Crawl - webdatacommons extractions. Check with Bizer et al. for progress.
 -[ ] Search for others.

## Library setup (1-time)

In [1]:
# Install necessary packages
!pip install feedparser requests beautifulsoup4
!pip install SPARQLWrapper

import pandas as pd
from io import StringIO

import feedparser
import requests
import sqlite3
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from bs4 import BeautifulSoup
import json
import re
from urllib.parse import urlparse, parse_qs, urljoin
import threading


## Database setup (1-time)

In [2]:
import sqlite3
import threading
from datetime import datetime
from urllib.parse import urljoin

# Initialize SQLite database with thread-safe connection
conn = sqlite3.connect('factcheck_alerts.db', check_same_thread=False)
cursor = conn.cursor()

# Initialize a reentrant threading lock for database operations
db_lock = threading.RLock()

# Create tables within the lock to ensure thread safety during setup
with db_lock:

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Feeds (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        feed_url TEXT UNIQUE,
        discovered_at TEXT,
        is_verified INTEGER DEFAULT 0
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS FetchTransactions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        fetched_at TEXT,
        crawl_duration REAL
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS WebFetches (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        transaction_id INTEGER,
        feed_url TEXT,
        fetch_url TEXT,
        fetched_at TEXT,
        content TEXT,
        request_headers TEXT,
        response_headers TEXT,
        status INTEGER,
        error TEXT,
        FOREIGN KEY(transaction_id) REFERENCES FetchTransactions(id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS DocProps (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        webfetch_id INTEGER,
        has_claim_review INTEGER,
        used_headless_browser INTEGER,
        is_extraction INTEGER,
        extracted_data TEXT,
        FOREIGN KEY(webfetch_id) REFERENCES WebFetches(id)
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        transaction_id INTEGER,
        log_time TEXT,
        message TEXT,
        FOREIGN KEY(transaction_id) REFERENCES FetchTransactions(id)
    )
    ''')

    conn.commit()


## Utility Functions

In [3]:
from bs4 import BeautifulSoup
import re

def update_is_verified_for_feeds(feed_urls):
  """Updates the `is_verified` column for a list of feeds.

  Args:
    feed_urls: A list of feed URLs, array.
  """
  with db_lock:
    for feed_url in feed_urls:
      cursor.execute("UPDATE Feeds SET is_verified = 1 WHERE feed_url = ?", (feed_url,))
    conn.commit()

def log_message(transaction_id, message):
    """Logs a message to the Logs table and prints it."""
    log_time = datetime.utcnow().isoformat()
    with db_lock:
        cursor.execute('INSERT INTO Logs (transaction_id, log_time, message) VALUES (?, ?, ?)',
                       (transaction_id, log_time, message))
        conn.commit()
    print(f"[{log_time}] {message}")

def store_fetch_transaction():
    """Stores a new fetch transaction and returns its ID."""
    fetched_at = datetime.utcnow().isoformat()
    with db_lock:
        cursor.execute('INSERT INTO FetchTransactions (fetched_at) VALUES (?)', (fetched_at,))
        conn.commit()
        return cursor.lastrowid

def update_crawl_duration(transaction_id, start_time):
    """Updates the crawl duration for a given transaction."""
    end_time = datetime.utcnow()
    duration = (end_time - start_time).total_seconds()
    with db_lock:
        cursor.execute('UPDATE FetchTransactions SET crawl_duration = ? WHERE id = ?',
                       (duration, transaction_id))
        conn.commit()

def store_webfetch(transaction_id, feed_url, fetch_url, content, req_headers, resp_headers, status, error):
    """Stores a webfetch record and returns its ID."""
    fetched_at = datetime.utcnow().isoformat()
    with db_lock:
        # Convert headers to standard dicts and ensure all values are serializable
        req_headers_dict = {k: v for k, v in req_headers.items()} if req_headers else {}
        resp_headers_dict = {k: v for k, v in resp_headers.items()} if resp_headers else {}

        # Handle any non-serializable data by converting to strings
        try:
            req_headers_json = json.dumps(req_headers_dict)
        except TypeError:
            req_headers_json = json.dumps({k: str(v) for k, v in req_headers_dict.items()})

        try:
            resp_headers_json = json.dumps(resp_headers_dict)
        except TypeError:
            resp_headers_json = json.dumps({k: str(v) for k, v in resp_headers_dict.items()})

        cursor.execute('''
            INSERT INTO WebFetches
            (transaction_id, feed_url, fetch_url, fetched_at, content, request_headers, response_headers, status, error)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (transaction_id, feed_url, fetch_url, fetched_at, content,
              req_headers_json,
              resp_headers_json,
              status, error))
        conn.commit()
        return cursor.lastrowid

def store_docprops(webfetch_id, has_claim_review, used_headless_browser, is_extraction, extracted_data):
    """Stores document properties for a webfetch."""
    with db_lock:
        cursor.execute('''
            INSERT INTO DocProps
            (webfetch_id, has_claim_review, used_headless_browser, is_extraction, extracted_data)
            VALUES (?, ?, ?, ?, ?)
        ''', (webfetch_id, has_claim_review, used_headless_browser, is_extraction, extracted_data))

        if has_claim:
              with db_lock:
                  cursor.execute('UPDATE Feeds SET is_verified = 1 WHERE feed_url = ?', (feed_url,))
                  conn.commit()


        conn.commit()

def extract_claim_review(content):
    """Extracts ClaimReview data from the content."""
    soup = BeautifulSoup(content, 'html.parser')
    has_claim = 'ClaimReview' in content
    extracted = False
    extracted_data = None

    if has_claim:
        # Extract JSON-LD
        scripts = soup.find_all('script', type='application/ld+json')
        claim_reviews = []
        for script in scripts:
            try:
                data = json.loads(script.string)
                if isinstance(data, list):
                    items = data
                else:
                    items = [data]
                for item in items:
                    if item.get('@type') == 'ClaimReview':
                        claim_reviews.append(item)
            except (json.JSONDecodeError, TypeError):
                continue

        # Extract Microdata (basic example)
        microdata = []
        for item in soup.find_all(attrs={'itemtype': re.compile('ClaimReview')}):
            microdata.append(str(item))

        if claim_reviews or microdata:
            extracted = True
            extracted_data = json.dumps({
                "json_ld": claim_reviews,
                "microdata": microdata
            })

    return has_claim, extracted, extracted_data


In [4]:
  import requests
from urllib.parse import urlparse, urljoin
from xml.etree import ElementTree

def parse_sitemap(sitemap_url, namespace, discovered_sitemaps=None, max_depth=5, current_depth=0):
    """Recursively parses sitemaps and collects all URLs with a depth limit."""
    if discovered_sitemaps is None:
        discovered_sitemaps = set()

    if current_depth > max_depth:
        log_message(None, f"Max sitemap recursion depth reached at {sitemap_url}")
        return

    try:
        response = requests.get(sitemap_url, timeout=10)
        if response.status_code == 200:
            tree = ElementTree.fromstring(response.content)
            # Check if it's a sitemap index
            sitemap_tags = tree.findall('ns:sitemap/ns:loc', namespace)
            if sitemap_tags:
                for sitemap in sitemap_tags:
                    sitemap_loc = sitemap.text
                    if sitemap_loc not in discovered_sitemaps:
                        discovered_sitemaps.add(sitemap_loc)
                        log_message(None, f"Recursively parsing sitemap: {sitemap_loc}")
                        parse_sitemap(sitemap_loc, namespace, discovered_sitemaps, max_depth, current_depth + 1)
            else:
                # It's a regular sitemap
                for url_elem in tree.findall('ns:url/ns:loc', namespace):
                    yield url_elem.text
    except Exception as e:
        log_message(None, f"Error parsing sitemap {sitemap_url}: {e}")

def discover_feeds_from_claim_reviews():
    """Discovers new RSS/Atom feeds and sitemap URLs from existing ClaimReview documents."""
    with db_lock:
        cursor.execute('''
            SELECT DISTINCT wf.fetch_url
            FROM WebFetches wf
            JOIN DocProps dp ON wf.id = dp.webfetch_id
            WHERE dp.has_claim_review = 1
        ''')
        fetch_urls = cursor.fetchall()

    base_urls = set()
    for (url,) in fetch_urls:
        parsed_url = urlparse(url)
        base_url = f"{parsed_url.scheme}://{parsed_url.netloc}/"
        base_urls.add(base_url)

    log_message(None, f"Discovered {len(base_urls)} unique base sites from ClaimReview documents.")

    discovered_feeds = set()

    for base_url in base_urls:
        log_message(None, f"Probing base site: {base_url}")

        # 1. Attempt to find and parse sitemap.xml recursively with depth limit
        sitemap_url = urljoin(base_url, 'sitemap.xml')
        try:
            response = requests.get(sitemap_url, timeout=10)
            if response.status_code == 200:
                log_message(None, f"Found sitemap at {sitemap_url}")
                for page_url in parse_sitemap(sitemap_url, {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}):
                    # Optionally, parse each page_url to find feeds
                    pass  # Implement if needed
        except Exception as e:
            log_message(None, f"No sitemap found at {sitemap_url}: {e}")

        # 2. Search for RSS/Atom feeds in the homepage with expanded criteria
        try:
            response = requests.get(base_url, timeout=10)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                # Look for various types of feed links
                feed_links = soup.find_all('link', rel=['alternate', 'alternate stylesheet'], type=['application/rss+xml', 'application/atom+xml', 'application/feed+json'])
                for link in feed_links:
                    feed_href = link.get('href')
                    if feed_href:
                        feed_url = urljoin(base_url, feed_href)
                        discovered_feeds.add(feed_url)
                        log_message(None, f"Discovered feed: {feed_url}")
        except Exception as e:
            log_message(None, f"Error fetching homepage {base_url}: {e}")

        # 3. Probe common feed URLs
        common_feed_paths = ['rss', 'feed', 'rss.xml', 'atom.xml', 'feeds/posts/default']
        for path in common_feed_paths:
            feed_url = urljoin(base_url, path)
            try:
                response = requests.get(feed_url, timeout=10)
                if response.status_code == 200 and 'xml' in response.headers.get('Content-Type', ''):
                    discovered_feeds.add(feed_url)
                    log_message(None, f"Discovered common feed URL: {feed_url}")
            except Exception as e:
                log_message(None, f"Error probing common feed URL {feed_url}: {e}")

    # 4. Store discovered feeds in the Feeds table
    new_feeds = 0
    with db_lock:
        for feed in discovered_feeds:
            try:
                cursor.execute('INSERT INTO Feeds (feed_url, discovered_at) VALUES (?, ?)',
                               (feed, datetime.utcnow().isoformat()))
                new_feeds += 1
                log_message(None, f"Added new feed to database: {feed}")
            except sqlite3.IntegrityError:
                # Feed already exists
                log_message(None, f"Feed already exists in database: {feed}")
        conn.commit()

    log_message(None, f"Feed discovery completed. {new_feeds} new feeds added.")


In [5]:
import feedparser
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse, parse_qs, urljoin

def fetch_and_process_feed(feed_url, transaction_id):
    """Fetches a feed, parses entries, extracts ClaimReview data, and stores them."""
    log_message(transaction_id, f"Parsing feed: {feed_url}")
    try:
        feed = feedparser.parse(feed_url)
        if not feed.entries:
            log_message(transaction_id, f"No entries found in feed: {feed_url}")
            return
        for entry in feed.entries:
            fetch_url = entry.get('link')
            if not fetch_url:
                continue  # Skip entries without a link

            # Check if the URL is a Google proxy URL
            parsed_fetch_url = urlparse(fetch_url)
            if parsed_fetch_url.netloc == 'www.google.com' and parsed_fetch_url.path == '/url':
                query_params = parse_qs(parsed_fetch_url.query)
                actual_urls = query_params.get('url')
                if actual_urls:
                    fetch_url = actual_urls[0]  # Use the first actual URL

            # Fetch the content of the fetch_url
            try:
                response = requests.get(fetch_url, timeout=10)
                status = response.status_code
                content = response.text if response.status_code == 200 else ""
                req_headers = dict(response.request.headers)
                resp_headers = dict(response.headers)
                error = "" if response.status_code == 200 else f"HTTP {response.status_code}"
            except Exception as e:
                status = None
                content = ""
                req_headers = {}
                resp_headers = {}
                error = str(e)

            # Store the webfetch
            webfetch_id = store_webfetch(
                transaction_id,
                feed_url,
                fetch_url,
                content,
                req_headers,
                resp_headers,
                status,
                error
            )

            if content:
                has_claim, extracted, extracted_data = extract_claim_review(content)
                store_docprops(
                    webfetch_id,
                    int(has_claim),
                    0,  # used_headless_browser initially no
                    int(extracted),
                    extracted_data
                )

                # If ClaimReview is found, mark the feed as verified
                if has_claim:
                    with db_lock:
                        cursor.execute('UPDATE Feeds SET is_verified = 1 WHERE feed_url = ?', (feed_url,))
                        conn.commit()
    except Exception as e:
        log_message(transaction_id, f"Error parsing feed {feed_url}: {e}")

def fetch_initial_feeds():
    """Fetches only verified feeds and processes their entries."""
    transaction_id = store_fetch_transaction()
    log_message(transaction_id, "Fetch transaction started.")

    with db_lock:
        cursor.execute('SELECT feed_url FROM Feeds WHERE is_verified = 1')
        verified_feeds = [row[0] for row in cursor.fetchall()]

    if not verified_feeds:
        log_message(transaction_id, "No verified feeds to fetch.")
        return

    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(fetch_and_process_feed, feed_url, transaction_id) for feed_url in verified_feeds]
        for future in as_completed(futures):
            try:
                future.result()
            except Exception as e:
                log_message(transaction_id, f"Error in feed fetching thread: {e}")

    log_message(transaction_id, "Initial feed fetching completed.")

def discover_feeds_from_claim_reviews():
    """Discovers new RSS/Atom feeds from existing ClaimReview documents."""
    with db_lock:
        cursor.execute('''
            SELECT DISTINCT wf.fetch_url
            FROM WebFetches wf
            JOIN DocProps dp ON wf.id = dp.webfetch_id
            WHERE dp.has_claim_review = 1
        ''')
        fetch_urls = cursor.fetchall()

    base_urls = set()
    for (url,) in fetch_urls:
        parsed_url = urlparse(url)
        base_url = f"{parsed_url.scheme}://{parsed_url.netloc}/"
        base_urls.add(base_url)

    log_message(None, f"Discovered {len(base_urls)} unique base sites from ClaimReview documents.")

    discovered_feeds = set()

    for base_url in base_urls:
        log_message(None, f"Probing base site: {base_url}")

        # 1. Attempt to find and parse sitemap.xml recursively with depth limit
        sitemap_url = urljoin(base_url, 'sitemap.xml')
        try:
            response = requests.get(sitemap_url, timeout=10)
            if response.status_code == 200:
                log_message(None, f"Found sitemap at {sitemap_url}")
                for page_url in parse_sitemap(sitemap_url, {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}):
                    # Optionally, parse each page_url to find feeds
                    pass  # Implement if needed
        except Exception as e:
            log_message(None, f"No sitemap found at {sitemap_url}: {e}")

        # 2. Search for RSS/Atom feeds in the homepage with expanded criteria
        try:
            response = requests.get(base_url, timeout=10)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                # Look for various types of feed links
                feed_links = soup.find_all('link', rel=['alternate', 'alternate stylesheet'], type=['application/rss+xml', 'application/atom+xml', 'application/feed+json'])
                for link in feed_links:
                    feed_href = link.get('href')
                    if feed_href:
                        feed_url = urljoin(base_url, feed_href)
                        discovered_feeds.add(feed_url)
                        log_message(None, f"Discovered feed: {feed_url}")
        except Exception as e:
            log_message(None, f"Error fetching homepage {base_url}: {e}")

        # 3. Probe common feed URLs
        common_feed_paths = ['rss', 'feed', 'rss.xml', 'atom.xml', 'feeds/posts/default']
        for path in common_feed_paths:
            feed_url = urljoin(base_url, path)
            try:
                response = requests.get(feed_url, timeout=10)
                if response.status_code == 200 and 'xml' in response.headers.get('Content-Type', ''):
                    discovered_feeds.add(feed_url)
                    log_message(None, f"Discovered common feed URL: {feed_url}")
            except Exception as e:
                log_message(None, f"Error probing common feed URL {feed_url}: {e}")

    # 4. Store discovered feeds in the Feeds table
    new_feeds = 0
    with db_lock:
        for feed in discovered_feeds:
            try:
                cursor.execute('INSERT INTO Feeds (feed_url, discovered_at) VALUES (?, ?)',
                               (feed, datetime.utcnow().isoformat()))
                new_feeds += 1
                log_message(None, f"Added new feed to database: {feed}")
            except sqlite3.IntegrityError:
                # Feed already exists
                log_message(None, f"Feed already exists in database: {feed}")
        conn.commit()

    log_message(None, f"Feed discovery completed. {new_feeds} new feeds added.")

def summarize_db():
    """Provides a summary of the SQLite database state with limited details."""
    with db_lock:
        # Total Counts
        cursor.execute('SELECT COUNT(*) FROM FetchTransactions')
        total_transactions = cursor.fetchone()[0]

        cursor.execute('SELECT COUNT(*) FROM Feeds')
        total_feeds = cursor.fetchone()[0]

        cursor.execute('SELECT COUNT(*) FROM WebFetches')
        total_webfetches = cursor.fetchone()[0]

        cursor.execute('SELECT COUNT(*) FROM DocProps WHERE has_claim_review = 1')
        total_claim_reviews = cursor.fetchone()[0]

        cursor.execute('SELECT COUNT(*) FROM Logs')
        total_logs = cursor.fetchone()[0]

        # Recent Fetch Transactions
        cursor.execute('''
            SELECT id, fetched_at, crawl_duration
            FROM FetchTransactions
            ORDER BY fetched_at DESC
            LIMIT 5
        ''')
        recent_transactions = cursor.fetchall()

        # Recent Feeds
        cursor.execute('''
            SELECT feed_url, discovered_at
            FROM Feeds
            ORDER BY discovered_at DESC
            LIMIT 5
        ''')
        recent_feeds = cursor.fetchall()

        # Recent WebFetches (Reduced Detail)
        cursor.execute('''
            SELECT fetch_url, status
            FROM WebFetches
            ORDER BY fetched_at DESC
            LIMIT 5
        ''')
        recent_webfetches = cursor.fetchall()

        # Recent ClaimReview Documents
        cursor.execute('''
            SELECT wf.fetch_url, dp.extracted_data, wf.fetched_at
            FROM WebFetches wf
            JOIN DocProps dp ON wf.id = dp.webfetch_id
            WHERE dp.has_claim_review = 1
            ORDER BY wf.fetched_at DESC
            LIMIT 5
        ''')
        recent_claim_reviews = cursor.fetchall()

        # Recent Logs
        cursor.execute('''
            SELECT log_time, message
            FROM Logs
            ORDER BY log_time DESC
            LIMIT 10
        ''')
        recent_logs = cursor.fetchall()

    # Display Summary
    print("=== SQLite Database Summary ===\n")

    print(f"Total Fetch Transactions: {total_transactions}")
    print(f"Total Feeds: {total_feeds}")
    print(f"Total WebFetches: {total_webfetches}")
    print(f"Total ClaimReview Documents: {total_claim_reviews}")
    print(f"Total Logs: {total_logs}\n")

    print("=== Recent Fetch Transactions (Last 5) ===")
    for txn in recent_transactions:
        print(f"ID: {txn[0]}, Fetched At: {txn[1]}, Duration: {txn[2]} seconds")
    print()

    print("=== Recent Feeds (Last 5) ===")
    for feed in recent_feeds:
        print(f"Feed URL: {feed[0]}, Discovered At: {feed[1]}")
    print()

    print("=== Recent WebFetches (Last 5) ===")
    for wf in recent_webfetches:
        print(f"URL: {wf[0]}, Status: {wf[1]}")
    print()

    print("=== Recent ClaimReview Documents (Last 5) ===")
    for cr in recent_claim_reviews:
        print(f"URL: {cr[0]}, Fetched At: {cr[2]}")
        print(f"Extracted Data: {cr[1]}\n")

    print("=== Recent Logs (Last 10) ===")
    for log in recent_logs:
        print(f"[{log[0]}] {log[1]}")
    print()

# Crawl Feeds, Schema, Sitemaps etc.

### 1-time seeding Feed database

We start with two feeds populated with article alerts mentioning "Fact check" or "Factcheck". Add alternative RSS feeds here.

In [7]:
# 1-time run

update_is_verified_for_feeds(["https://www.google.co.uk/alerts/feeds/06076043609717911844/10774177467254939926",
"https://www.google.co.uk/alerts/feeds/06076043609717911844/18347646755010161793"])

### Regular re-crawl of feeds

In [ ]:
# Run the feed discovery process


# Step 1: Fetch and process initial feeds
fetch_initial_feeds()

# Step 2: Discover new feeds based on existing ClaimReview documents
discover_feeds_from_claim_reviews()

# Step 3: Summarize the database
summarize_db()


In [ ]:
with db_lock:
    cursor.execute("PRAGMA table_info(Feeds)")
    columns = [info[1] for info in cursor.fetchall()]
    print("Feeds Table Columns:", columns)


In [ ]:
summarize_db()

## Examples: Adding more seed URLs for Factcheck sites

In [ ]:
initial_feeds = [
    "https://www.snopes.com/feed/"
]

with db_lock:
    for feed in initial_feeds:
        try:
            cursor.execute('INSERT INTO Feeds (feed_url, discovered_at, is_verified) VALUES (?, ?, ?)',
                           (feed, datetime.utcnow().isoformat(), 1))
            print(f"Added and verified feed: {feed}")
        except sqlite3.IntegrityError:
            print(f"Feed already exists: {feed}")
    conn.commit()


# IFCN Signatories


In [ ]:
csv = """Name,Country,Verification Date,Profile URL
20 Minutes Fake off,France,8/23/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/20-minutes-fake-off
ABS-CBN Corporation,Philippines,3/27/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/abs-cbn-corporation
AFP fact checking,France,8/23/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/afp-fact-checking
AP Fact Check,United States,8/23/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/ap-fact-check
APA - Austria Presse Agentur,Austria,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/apa-austria-presse-agentur
Action for Democratic Society (ADS) / hibrid.info,Kosovo,10/16/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/action-for-democratic-society-ads-hibridinfo
Africa Check,South Africa,1/19/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/africa-check
AkhbarMeter,Egypt,11/3/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/akhbarmeter
Animal Político - El Sabueso,Mexico,8/23/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/animal-político---el-sabueso
Annie Lab,Hong Kong,9/13/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/annie-lab
Aos Fatos,Brazil,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/aos-fatos
Australian Associated Press,Australia,3/12/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/australian-associated-press
BOOM,India,1/20/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/boom
Bayerischer Rundfunk - BR24 #Faktenfuchs,Germany,11/3/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/bayerischer-rundfunk-br24-faktenfuchs
Belarusian Investigative Center,Czech Republic,6/18/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/belarusian-investigative-center
Bolivia Verifica,"Bolivia, Plurinational State of",12/8/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/bolivia-verifica
CORRECTIV,Germany,1/19/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/correctiv
Cek Fakta - Liputan 6,Indonesia,1/19/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/cek-fakta-liputan-6
Check Your Fact,United States,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/check-your-fact
Chequeado,Argentina,5/19/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/chequeado
Colombiacheck,Colombia,3/27/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/colombiacheck
Cotejo.Info,"Venezuela, Bolivarian Republic of",3/5/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/cotejoinfo
DELFI Melo Detektorius,Lithuania,11/3/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/delfi-melo-detektorius
Demagog.cz,Czech Republic,5/19/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/demagogcz
Deutsche Welle,Germany,3/19/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/deutsche-welle
Digital Forensics Research and Analytics Centre (D-FRAC),India,3/27/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/digital-forensics-research-and-analytics-centre-d-frac
Doğrula,Turkey,1/31/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/dogrula
Doğruluk Payı,Turkey,12/7/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/dogruluk-payi
EFE Verifica,Spain,5/10/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/efe-verifica
Ecuador Chequea,Ecuador,11/2/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/ecuador-chequea
Eesti Päevaleht / Ekspress Meedia,Estonia,12/7/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/eesti-paevaleht-ekspress-meedia
El Detector / Univision Noticias,United States,12/8/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/el-detector-univision-noticias
Ellinika Hoaxes (Greek Hoaxes),Greece,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/ellinika-hoaxes-greek-hoaxes
Estadão Verifica,Brazil,12/18/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/estadao-verifica
FACTLY MEDIA & RESEARCH,India,5/10/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factly-media-research
Fact Check Cyprus,Cyprus,8/23/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/fact-check-cyprus
Fact Check Zimbabwe,Zimbabwe,10/24/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/fact-check-zimbabwe
FactCheck Georgia,Georgia,3/19/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factcheck-georgia
FactCheck.org,United States,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factcheckorg
FactCheckNI,United Kingdom,1/19/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/factcheckni
FactCrescendo,India,5/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factcrescendo
FactReview,Greece,6/18/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factreview
FactWatch,Bangladesh,6/18/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factwatch
Facta,Italy,11/3/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/facta
Factcheck Lab,Hong Kong,6/18/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factcheck-lab
Factcheck.bg,Bulgaria,8/23/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/factcheckbg
Factchequeado.com,United States,9/30/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/factchequeado.com
Factnameh,Canada,9/30/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/factnameh
Faktisk.no,Norway,11/3/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/faktiskno
Faktograf-udruga za informiranu javnost,Croatia,6/4/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/faktograf-udruga-za-informiranu-javnost
Faktoje.al,Albania,8/23/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/faktojeal
Fast Check CL,Chile,2/14/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/fast-check-cl
Ferret Fact Service,United Kingdom,2/1/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/ferret-fact-service
First Check,India,5/10/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/first-check
Full Fact,United Kingdom,1/19/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/full-fact
Fundacja "Przeciwdziałamy Dezinformacji",Poland,5/10/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/fundacja-przeciwdzialamy-dezinformacji
Funky Citizens,Romania,11/2/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/funky-citizens
Greece Fact Check,Greece,1/23/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/greece-fact-check
INTERNEWS KOSOVA,Kosovo,10/16/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/internews-kosova
Infoveritas,Spain,11/10/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/infoveritas
Istinomer,Serbia,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/istinomer
Istinomjer,Bosnia and Herzegovina,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/istinomjer
Knack Magazine Roularta Media Group,Belgium,5/10/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/knack-magazine-roularta-media-group
Källkritikbyrån,Sweden,9/13/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/kallkritikbyran
La Silla Vacía,Colombia,9/13/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/la-silla-vacía
Lead Stories,United States,12/8/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/lead-stories
Les Surligneurs,France,8/23/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/les-surligneurs
Litmus,Japan,8/2/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/litmus
Lupa,Brazil,1/19/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/lupa
MAFINDO,Indonesia,3/27/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/mafindo
Mala Espina Check,Chile,5/10/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/mala-espina-check
Maldita.es,Spain,11/3/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/maldita.es
MediaWise,United States,8/23/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/mediawise
Medizin transparent - Universität für Weiterbildung Krems (Donau-Universität Krems),Austria,1/31/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/medizin-transparent-universitat-fur-weiterbildung-krems-donau-universitat-krems
MindaNews,Philippines,8/26/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/mindanews
Myth Detector,Georgia,5/10/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/myth-detector
Nest center for Journalism Innovation and Development NGO,Mongolia,8/23/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/nest-center-for-journalism-innovation-and-development-ngo
NewsMobile,India,9/18/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/newsmobile
Newsmeter (Fifth Estate Digital Private Limited),India,6/18/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/newsmeter-fifth-estate-digital-private-limited
Newtral,Spain,5/19/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/newtral
Observador - Fact Check,Portugal,3/13/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/observador-fact-check
Ocote,Guatemala,3/8/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/ocote
Open.online,Italy,11/3/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/openonline
Oštro center for investigative journalism in the Adriatic region,Slovenia,1/19/2024,https://ifcncodeofprinciples.poynter.org/signatories/profile/ostro-center-for-investigative-journalism-in-the-adriatic-region
PA Media,United Kingdom,9/30/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/pa-media
Pagella Politica,Italy,9/13/2023,https://ifcncodeofprinciples.poynter.org/signatories/profile/pagella-politica
PesaCheck,Kenya,6/18/2024,https://ifc"""

In [ ]:
# prompt: Python funxtion to update the CSV to replace broken URLs with this format: https://ifcncodeofprinciples.poynter.org/profile/abs-cbn-corporation i.e. drop the /signatories from path each time

import pandas as pd
import re

def update_csv_urls(csv):
  """Read in and tidy the IFCN data."""
  csv_df = pd.read_csv(StringIO(csv))

  # Apply the correction to the 'Profile URL' column
  csv_df['Profile URL'] = csv_df['Profile URL'].apply(lambda x: re.sub(r'/signatories/', '/', x))

  # Convert the DataFrame back to CSV string
  # updated_csv_data = df.to_csv(index=False)
  return csv_df

# Example usage:
# Assuming 'csv' is the variable containing the CSV data

csv_df = update_csv_urls(csv)

In [ ]:
print(csv_df.describe())
print(csv_df.head())

# Countries with most fact checkers:
csv_df['Country'].value_counts().head()


In [ ]:
# prompt: list URLs in dataframe for entries where Country is United Kingdom and United States

# Filter the DataFrame for entries where Country is "United Kingdom" or "United States"
filtered_df = df[df['Country'].isin(['United Kingdom', 'United States'])]

# Extract the "Profile URL" column from the filtered DataFrame
urls = filtered_df['Profile URL'].tolist()

# Print the list of URLs
print(urls)


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def ifcn_profile_extract(url):
    """
    Fetches the web page at the given URL, extracts specific fields from the
    <header class="mb-5"> section, and returns the data as a dictionary.

    Parameters:
        url (str): The URL of the web page to fetch and parse.

    Returns:
        dict: A dictionary containing the extracted fields.
    """
    profile = {}

    try:
        # Fetch the web page content
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raise an error for bad status codes
    except requests.RequestException as e:
        print(f"Error fetching URL {url}: {e}")
        return profile  # Return empty profile on failure

    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # Locate the <header> section with class 'mb-5'
    header = soup.find('header', class_='mb-5')
    if not header:
        print("Header with class 'mb-5' not found.")
        return profile

    # Extract Organization Name
    org_name_tag = header.find('h1', class_='mb-2')
    if org_name_tag and org_name_tag.find('strong'):
        profile['Organization Name'] = org_name_tag.find('strong').get_text(strip=True)
    else:
        profile['Organization Name'] = None

    # Define a mapping from label text to desired dictionary keys
    field_mapping = {
        'Country:': 'Country',
        'Language:': 'Language',
        'Issued on:': 'Issued On',
        'Expires on:': 'Expires On',
        'Badge shown on website:': 'Badge Shown on Website',
        'Website:': 'Website'
    }

    # Find all <p> tags within the header
    p_tags = header.find_all('p')
    for p in p_tags:
        strong_tag = p.find('strong')
        if strong_tag:
            label = strong_tag.get_text(strip=True)
            # The value is the text after the <strong> tag
            # Handle cases where the value might be within an <a> tag (e.g., Website)
            if label in field_mapping:
                key = field_mapping[label]
                # For 'Website:', extract the href from the <a> tag
                if label == 'Website:':
                    a_tag = p.find('a')
                    if a_tag and a_tag.has_attr('href'):
                        profile[key] = a_tag['href']
                    else:
                        # Fallback to the text if <a> tag is missing
                        profile[key] = strong_tag.next_sibling.strip() if strong_tag.next_sibling else None
                else:
                    # Extract the text after the <strong> tag
                    value = strong_tag.next_sibling
                    if value:
                        profile[key] = value.strip()
                    else:
                        profile[key] = None

    return profile


In [ ]:
# Example usage within your existing workflow
sample_url = "https://ifcncodeofprinciples.poynter.org/profile/ap-fact-check'"  # Replace with actual IFCN profile URL
profile = ifcn_profile_extract(sample_url)

if profile:
    print("Extracted IFCN Profile:")
    for key, value in profile.items():
        print(f"{key}: {value}")
else:
    print("No profile data extracted.")


# Wikidata as Factchecker repository

Extracting a list of members of IFCN [from wikidata](https://w.wiki/BEPR), then using that to look for sitemaps and feeds.

## Utility functions (wikidata-related)

In [9]:
# Utility functions for fetching factchecker info from Wikidata

import json
from SPARQLWrapper import SPARQLWrapper, JSON

def wikidata_ifcn_member_urls():
    """
    Submits a SPARQL query to Wikidata to retrieve IFCN members, their labels, and official websites.

    Returns:
        list of dict: Each dictionary contains 'member', 'memberLabel', and 'url' keys.
    """
    # Define the SPARQL endpoint and query
    sparql_endpoint = "https://query.wikidata.org/sparql"
    query = """
    SELECT ?member ?memberLabel ?url
    WHERE {
      # Q51698517 is the Wikidata ID for IFCN
      # P463 is the property for "member of"
      ?member wdt:P463 wd:Q51698517.

      # P856 is the property for "official website"
      OPTIONAL { ?member wdt:P856 ?url. }

      # This retrieves the label of the member in the user's language
      SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
    }
    """

    # Initialize the SPARQL wrapper
    sparql = SPARQLWrapper(sparql_endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    try:
        # Execute the query and convert the results to JSON
        results = sparql.query().convert()
    except Exception as e:
        print(f"An error occurred while querying Wikidata: {e}")
        return []

    # Process the results
    data = []
    for result in results["results"]["bindings"]:
        member = result.get("member", {}).get("value")
        memberLabel = result.get("memberLabel", {}).get("value")
        url = result.get("url", {}).get("value")

        data.append({
            "member": member,
            "memberLabel": memberLabel,
            "url": url
        })

    return data


# prompt: given a python array of dictionaries with a key 'url', scan each for feeds/sitemaps as defined earlier.

def find_feeds_sitemaps(data):
    """
    Scans a list of URLs or dictionaries for feeds and sitemaps.

    Args:
        data: A list of URLs (strings) or a list of dictionaries, each containing a 'url' key.

    Returns:
        A list of dictionaries with 'url', 'feed_urls', and 'sitemap_urls' keys.
    """
    results = []

    # Convert input to list of URLs if it's a list of strings
    if all(isinstance(item, str) for item in data):
        urls = data
    elif all(isinstance(item, dict) for item in data):
        urls = [item.get('url') for item in data if item.get('url')]
    else:
        raise ValueError("Input must be either a list of URLs or a list of dictionaries containing 'url' keys")

    for url in urls:
        feed_urls = []
        sitemap_urls = []
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()  # Raise an error for bad status codes
            soup = BeautifulSoup(response.text, 'html.parser')

            # Find feed URLs
            for link in soup.find_all('link'):
                if link.get('type') in ['application/rss+xml', 'application/atom+xml']:
                    feed_url = link.get('href')
                    if feed_url:
                        feed_urls.append(urljoin(url, feed_url))

            # Find sitemap URLs
            for link in soup.find_all('link'):
                if link.get('rel') == 'sitemap':
                    sitemap_url = link.get('href')
                    if sitemap_url:
                        sitemap_urls.append(urljoin(url, sitemap_url))

            # Handle cases where sitemap is in a different format
            for a in soup.find_all('a'):
                href = a.get('href')
                if href and 'sitemap' in href:
                    sitemap_urls.append(urljoin(url, href))

        except requests.RequestException as e:
            print(f"Error fetching URL {url}: {e}")

        results.append({
            'url': url,
            'feed_urls': feed_urls,
            'sitemap_urls': sitemap_urls
        })
    return results



# prompt: python - every entry in an array of dictionaries, combine the base_href derrived from main 'url' property with relative links in 'feed_urls'' property. Consolidate into a flat list of absolute feed URIs.

def combine_urls(data):
  """
  Combines base href derived from main 'url' property with relative links in 'feed_urls' property.
  Consolidates into a flat list of absolute feed URIs.

  Args:
    data: A list of dictionaries with 'url' and 'feed_urls' keys.

  Returns:
    A list of absolute feed URIs.
  """
  absolute_feed_urls = []
  for item in data:
    base_url = item.get('url')
    if not base_url:
      continue

    feed_urls = item.get('feed_urls', [])
    for feed_url in feed_urls:
      if not feed_url.startswith('http'):
        # Construct absolute URL
        absolute_feed_url = urljoin(base_url, feed_url)
        absolute_feed_urls.append(absolute_feed_url)
      else:
        absolute_feed_urls.append(feed_url)

  return absolute_feed_urls

## Wikidata-to-Feeds extraction

In [10]:
wd_urls = wikidata_ifcn_member_urls()

In [ ]:
wd_urls

In [11]:
fc_feedinfos = find_feeds_sitemaps(wd_urls)

In [13]:
fc_feedinfos

[{'url': 'https://www.politifact.com/',
  'feed_urls': ['https://www.politifact.com/rss/all/',
   'https://www.politifact.com/rss/factchecks/'],
  'sitemap_urls': []},
 {'url': 'http://www.chequeado.com/',
  'feed_urls': ['https://chequeado.com/feed/?post_type=nota'],
  'sitemap_urls': []},
 {'url': 'http://demagog.cz/',
  'feed_urls': ['http://demagog.cz/rss/index.atom'],
  'sitemap_urls': []},
 {'url': 'http://www.africacheck.org',
  'feed_urls': [],
  'sitemap_urls': ['http://www.africacheck.org/sitemap']},
 {'url': 'https://fr.africacheck.org/',
  'feed_urls': ['https://affirmado.com/feed/',
   'https://affirmado.com/comments/feed/'],
  'sitemap_urls': []},
 {'url': 'https://pagellapolitica.it/', 'feed_urls': [], 'sitemap_urls': []},
 {'url': 'https://fullfact.org/',
  'feed_urls': ['https://fullfact.org/feed/all/'],
  'sitemap_urls': []},
 {'url': 'https://www.ellinikahoaxes.gr/',
  'feed_urls': [],
  'sitemap_urls': []},
 {'url': 'https://www.faktyoxla.info/', 'feed_urls': [], 's

In [14]:

factchecker_feeds = combine_urls(fc_feedinfos)


In [15]:
factchecker_feeds

['https://www.politifact.com/rss/all/',
 'https://www.politifact.com/rss/factchecks/',
 'https://chequeado.com/feed/?post_type=nota',
 'http://demagog.cz/rss/index.atom',
 'https://affirmado.com/feed/',
 'https://affirmado.com/comments/feed/',
 'https://fullfact.org/feed/all/']

In [28]:
update_is_verified_for_feeds(factchecker_feeds)

In [27]:
manual_list_en_factcheckers = ["https://www.poynter.org/mediawise/", "https://www.poynter.org/mediawise/", "https://leadstories.com/", "https://fullfact.org/", "https://theferret.scot/fact-check/", "https://factcheckni.org/", "https://www.factcheck.org/", "https://checkyourfact.com/"]

manual_feedlist = find_feeds_sitemaps(manual_list_en_factcheckers)
manual_feedlist_as_urls = list(set(item['url'] for item in manual_feedlist if 'url' in item))
update_is_verified_for_feeds(manual_feedlist_as_urls)

Error fetching URL https://www.poynter.org/mediawise/: 403 Client Error: Forbidden for url: https://www.poynter.org/mediawise/
Error fetching URL https://www.poynter.org/mediawise/: 403 Client Error: Forbidden for url: https://www.poynter.org/mediawise/


In [29]:
with db_lock:
        df = pd.read_sql_query("SELECT feed_url, is_verified FROM Feeds", conn)
        df['is_verified'] = df['is_verified'].map({1: True, 0: False})
df

,feed_url,is_verified
0,https://www.factcheck.org/rss.xml,True
1,https://www.politifact.com/rss/,True
2,https://www.snopes.com/feed/,True
3,https://www.snopes.com/feed,False


# Debug Misc (noise, mostly)

In [ ]:
# Inspect the feedlist
print("Feedlist Type:", type(feedlist))
print("Number of Feeds:", len(feedlist))
print("Sample Feeds:", feedlist[:5])  # Print first 5 feeds for inspection

# Check types of individual feed URLs
for i, feed in enumerate(feedlist[:5], start=1):
    print(f"Feed {i} Type:", type(feed))
    print(f"Feed {i} Content:", feed)